# Phase 0: setup and verification
Re-runnable. Creates (if missing) the catalog, schemas, Volume and 9 control tables, then checks every Phase 0 component:
unit tests, warehouse connection and read-only guard, stage runner smoke test (control table + Volume + MLflow),
LLM tool-calling loop on `databricks-claude-sonnet-5-5`, and GitHub access through the secret scope.
All setup statements are create-only (`IF NOT EXISTS`).

In [0]:
%pip install -q databricks-sql-connector sqlglot pydantic pyyaml openai pytest

In [0]:
dbutils.library.restartPython()

In [0]:
import os, sys, subprocess, uuid
sys.dont_write_bytecode = True  # workspace files cannot hold __pycache__

# Repo root: this notebook lives in <root>/setup/
nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
ROOT = os.path.dirname(os.path.dirname("/Workspace" + nb_path if not nb_path.startswith("/Workspace") else nb_path))
assert os.path.exists(f"{ROOT}/databricks.yml"), f"repo root not found: {ROOT}"
CATALOG = "semantic_builder_poc"
WAREHOUSE_ID = "5a6ef6635baf81c5"
LLM_ENDPOINT = "databricks-claude-sonnet-5-5"
SECRET_SCOPE = "semantic_builder_poc"
GIT_REPO, GIT_BRANCH = "aryanshreni/Enrich_Agent", "main"
sys.path.insert(0, ROOT)
results = {}
print(ROOT)

In [0]:
def statements(path):
    text = "\n".join(l for l in open(path).read().splitlines() if not l.strip().startswith("--"))
    return [s.strip() for s in text.split(";") if s.strip()]

for f in ["setup/00_setup.sql", "setup/01_control_tables.sql"]:
    for s in statements(f"{ROOT}/{f}"):
        spark.sql(s)
    print("ran", f)

tables = spark.sql(f"SELECT table_name FROM {CATALOG}.information_schema.tables WHERE table_schema = 'control'").count()
volume = spark.sql(f"SELECT 1 FROM {CATALOG}.information_schema.volumes WHERE volume_schema = 'artifacts' AND volume_name = 'files'").count()
results["control tables (expect 9)"] = tables
results["artifact volume"] = volume == 1
display(spark.sql(f"SELECT table_name, comment FROM {CATALOG}.information_schema.tables WHERE table_schema = 'control' ORDER BY 1"))

In [0]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", f"{ROOT}/tests"],
                   capture_output=True, text=True, cwd="/tmp", env=dict(os.environ, PYTHONDONTWRITEBYTECODE="1"))
print(r.stdout[-2000:], r.stderr[-1000:])
results["unit tests"] = r.returncode == 0

In [0]:
from tools.readonly_sql import ReadOnlySql, ReadOnlyViolation

sqlt = ReadOnlySql(WAREHOUSE_ID)
print(sqlt.run("SELECT current_user() AS user"))
try:
    sqlt.run(f"DROP TABLE {CATALOG}.control.sources")
    results["guard blocks DROP"] = False
except ReadOnlyViolation as e:
    print("blocked:", e)
    results["guard blocks DROP"] = True
results["warehouse connection"] = True

In [0]:
sys.path.insert(0, f"{ROOT}/jobs")
import stage_runner

smoke_run = f"smoke-{uuid.uuid4().hex[:8]}"
sys.argv = ["stage_runner.py", "--run-id", smoke_run, "--stage", "0", "--catalog", CATALOG,
            "--warehouse-id", WAREHOUSE_ID, "--llm-endpoint", LLM_ENDPOINT, "--secret-scope", SECRET_SCOPE,
            "--git-repo", GIT_REPO, "--git-branch", GIT_BRANCH, "--job-run-id", ""]
stage_runner.main()
row = spark.sql(f"SELECT status, artifact_path FROM {CATALOG}.control.stage_runs WHERE run_id = '{smoke_run}'").first()
print(row)
print(dbutils.fs.head(row.artifact_path))
results["stage runner smoke"] = row.status == "done"

In [0]:
import mlflow
from pydantic import BaseModel
from agents.common.llm import ToolAgent, Tool
from tools.uc_metadata import UcMetadata

mlflow.set_experiment("/Shared/semantic_builder_poc/stage_agents")
meta = UcMetadata(sqlt)
tools = [
    Tool("run_sql", "Run one read-only SQL query", {"type": "object", "properties": {"sql": {"type": "string"}}, "required": ["sql"]},
         lambda sql: sqlt.run(sql)),
    Tool("list_tables", "List tables in a schema", {"type": "object", "properties": {"catalog": {"type": "string"}, "schema": {"type": "string"}},
         "required": ["catalog", "schema"]}, lambda catalog, schema: meta.list_tables(catalog, schema)),
]

class Answer(BaseModel):
    table_count: int
    largest_column_table: str

with mlflow.start_run(run_name=f"{smoke_run}-llm"):
    ans = ToolAgent(endpoint=LLM_ENDPOINT, tools=tools, max_turns=10).run(
        'Use tools. Return only JSON: {"table_count": int, "largest_column_table": str}.',
        f"In {CATALOG}.control, how many tables are there and which has the most columns?", response_model=Answer)
expected = spark.sql(f"SELECT table_name FROM {CATALOG}.information_schema.columns WHERE table_schema = 'control' "
                     "GROUP BY table_name ORDER BY count(*) DESC LIMIT 1").first()[0]
print(ans, "| expected largest:", expected, "| trace:", mlflow.get_last_active_trace_id())
results["LLM tool loop"] = ans.table_count == 9 and ans.largest_column_table == expected

In [0]:
# Needs a fine-grained GitHub token (Contents: read and write, this repo only) stored as:
#   databricks secrets put-secret semantic_builder_poc github_token
from tools.git_commit import GitCommitter, read_token

try:
    info = GitCommitter(GIT_REPO, GIT_BRANCH, read_token(SECRET_SCOPE)).check_access()
    print(info)
    results["GitHub access"] = bool(info["can_push"])
except Exception as e:
    print("GitHub check not passing yet:", type(e).__name__, str(e)[:200])
    results["GitHub access"] = False

In [0]:
for k, v in results.items():
    ok = v is True or v == 9
    print(f"{'PASS' if ok else 'TODO'}  {k}: {v}")
print("\nRemaining outside this notebook: bundle deploy, set job run_as to the app service principal, "
      "grant the app SP ownership of the catalog and SELECT on each raw schema, run the smoke test from the app.")